# 01 — Getting started with PySpark

This notebook covers Phase 1 of the roadmap: creating a SparkSession, loading data, and basic DataFrame operations.

Before running: make sure you've generated the sample data with `python scripts/generate_data.py` from the project root.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# local[*] = run Spark on this machine, using all CPU cores
spark = (
    SparkSession.builder
    .appName("getting-started")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
# While a job runs, the Spark UI is at http://localhost:4040


## Load the sample data

Spark is *lazy*: `spark.read.csv` doesn't actually scan the whole file — work only happens when you call an **action** like `show()` or `count()`.


In [ ]:
DATA_DIR = "../data/generated"

orders = spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA_DIR}/products.csv", header=True, inferSchema=True)

orders.printSchema()
orders.show(5)


## Basic operations

`select`, `filter`, `withColumn`, and `orderBy` are **transformations** — they build up a plan without executing it. `show()` triggers execution.


In [ ]:
# Orders over $500, biggest first
big_orders = (
    orders
    .filter(F.col("order_total") > 500)
    .select("order_id", "customer_id", "order_total", "order_date")
    .orderBy(F.col("order_total").desc())
)

big_orders.show(10)
print(f"{big_orders.count()} orders over $500")


In [ ]:
# Add a derived column, then aggregate: orders per year
orders_with_year = orders.withColumn("year", F.year("order_date"))

orders_with_year.groupBy("year").count().orderBy("year").show()


## A taste of joins (Phase 2 preview)

Join orders to products and compute revenue by category.


In [ ]:
revenue_by_category = (
    orders
    .join(products, on="product_id", how="inner")
    .groupBy("category")
    .agg(
        F.round(F.sum("order_total"), 2).alias("revenue"),
        F.count("order_id").alias("num_orders"),
    )
    .orderBy(F.col("revenue").desc())
)

revenue_by_category.show()


## Your turn

Try these before moving to Phase 2 of the roadmap (see README):

1. How many distinct customers placed at least one order?
2. What's the average `order_total` per `discount` level?
3. Find the 5 most recent orders (`orderBy` on `order_date`).
4. Some orders reference customers that don't exist in `customers.csv` — how many? (Hint: `join` with `how="left_anti"`.)

When you're done, stop the session with the cell below.


In [ ]:
# 1. How many distinct customers placed at least one order?


In [ ]:
# 2. What's the average order_total per discount level?


In [ ]:
# 3. Find the 5 most recent orders (orderBy on order_date)


In [ ]:
# 4. Some orders reference customers that don't exist in customers.csv — how many?
# (Hint: join with how="left_anti".)


In [ ]:
spark.stop()
